# BioNeMo Framework kernel test

Built from `nvcr.io/nvidia/clara/bionemo-framework:2.4`. Examples adapted from the [BioNeMo Framework 2.6 ESM-2 Inference guide](https://docs.nvidia.com/bionemo-framework/2.6/user-guide/examples/bionemo-esm2/inference/) 

## 1. Basic sanity check -- kernel/GPU/BioNeMo import

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

import bionemo
print("BioNeMo imported OK:", bionemo.__file__)

## 2. Download a small pretrained ESM-2 checkpoint

Confirms BioNeMo's model-loading machinery and NGC connectivity work.

In [ ]:
from bionemo.core.data.load import load

checkpoint_path = load("esm2/8m:2.0")  # smallest ESM-2 variant, fastest to fetch
print(checkpoint_path)

## 3. Full inference example -- protein sequences to embeddings

In [ ]:
import os
import pandas as pd
import torch

work_dir = "/tmp/esm2_test"
os.makedirs(work_dir, exist_ok=True)

artificial_sequence_data = [
    "TLILGWSDKLGSLLNQLAIANESLGGGTIAVMAERDKEDMELDIGKMEFDFKGTSVI",
    "LYSGDHSTQGARFLRDLAENTGRAEYELLSLF",
    "GRFNVWLGGNESKIRQVLKAVKEIGVSPTLFAVYEKN",
]
df = pd.DataFrame(artificial_sequence_data, columns=["sequences"])
data_path = os.path.join(work_dir, "sequences.csv")
df.to_csv(data_path, index=False)

This is a Python kernel, so the `infer_esm2` command-line tool needs the `!` shell-escape prefix, not a bare call.

In [ ]:
!infer_esm2 --checkpoint-path {checkpoint_path} \
    --data-path {data_path} \
    --results-path {work_dir} \
    --micro-batch-size 3 \
    --num-gpus 1 \
    --precision "bf16-mixed" \
    --include-hiddens \
    --include-embeddings \
    --include-logits \
    --include-input-ids

## 4. Read back the results

In [ ]:
results = torch.load(f"{work_dir}/predictions__rank_0.pt")
for key, val in results.items():
    if val is not None:
        print(f"{key}\t{val.shape}")